# DCA Machine Grouping Analysis
This notebook creates an Step-product incidence matrix and prepares it for Direct Clustering Algorithm (DCA) analysis.

In [77]:
import pandas as pd
import numpy as np

In [78]:
# Load the data
steps_df = pd.read_csv('../Data/Steps.csv')
equipment_df = pd.read_csv('../Data/Equipment.csv')

In [79]:
# Parse the equipment names

equipment_list = equipment_df['Equipment'].tolist()
products = steps_df['Part'].tolist()

In [80]:
# Get all unique steps required across all products
all_steps = set()
for idx, row in steps_df.iterrows():
    for col in steps_df.columns[1:]:  # Skip the 'Part' column
        step = row[col]
        if pd.notna(step) and step != '':
            all_steps.add(step)

all_steps = sorted(list(all_steps))
print(f"Unique steps: {all_steps}")

Unique steps: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']


In [81]:
# dictionary mapping each product to the set of steps it requires
product_steps = {}
for idx, row in steps_df.iterrows():
    product = row['Part']
    steps_required = set()
    for col in steps_df.columns[1:]:  # Skip the 'Part' column
        step = row[col]
        if pd.notna(step) and step != '':
            steps_required.add(step)
    product_steps[product] = steps_required


# # dictionary mapping each equipment into steps it can be done
# equipment_capabilities = {}
# for idx, row in equipment_df.iterrows():
#     equipment = row['Equipment']
#     caps = set()
#     caps = set(ch for ch in str(equipment) if ch.isalpha() and ch.isupper())
#     equipment_capabilities[equipment] = caps

print(product_steps)

{'P1': {'J', 'I', 'C', 'A', 'D', 'B'}, 'P2': {'J', 'C', 'H', 'A', 'D'}, 'P3': {'J', 'I', 'C', 'D', 'B'}, 'P4': {'H', 'G', 'A', 'D', 'B'}, 'P5': {'C', 'B', 'I'}, 'P6': {'J', 'I', 'C', 'H', 'A', 'D', 'B'}, 'P7': {'J', 'I', 'C', 'F', 'E'}, 'P8': {'J', 'H', 'E', 'G'}, 'P9': {'J', 'I', 'F', 'E', 'G'}, 'P10': {'F', 'E', 'G', 'I'}, 'P11': {'J', 'G', 'E', 'I'}, 'P12': {'F', 'J', 'G', 'E'}, 'P13': {'I', 'H', 'F', 'E', 'G'}, 'P14': {'G', 'H', 'E'}, 'P15': {'J', 'F', 'H', 'E'}, 'P16': {'F', 'H', 'J', 'I'}, 'P17': {'L', 'M', 'K'}, 'P18': {'L', 'M', 'K'}, 'P19': {'L', 'M'}, 'P20': {'L', 'M', 'K'}}


In [82]:
# create the relationship matrix 
matrix = np.zeros((len(product_steps), len(all_steps)), dtype=int)
# This creates (20 products × 13 steps) matrix ✓

for i, (part, req) in enumerate(product_steps.items()):  
    for j, step in enumerate(all_steps):                 
        if step in req:
            matrix[i, j] = 1 

incidence_df = pd.DataFrame(matrix, index=products, columns=all_steps)
# transposed_matrix = incidence_df.T

print(incidence_df)

     A  B  C  D  E  F  G  H  I  J  K  L  M
P1   1  1  1  1  0  0  0  0  1  1  0  0  0
P2   1  0  1  1  0  0  0  1  0  1  0  0  0
P3   0  1  1  1  0  0  0  0  1  1  0  0  0
P4   1  1  0  1  0  0  1  1  0  0  0  0  0
P5   0  1  1  0  0  0  0  0  1  0  0  0  0
P6   1  1  1  1  0  0  0  1  1  1  0  0  0
P7   0  0  1  0  1  1  0  0  1  1  0  0  0
P8   0  0  0  0  1  0  1  1  0  1  0  0  0
P9   0  0  0  0  1  1  1  0  1  1  0  0  0
P10  0  0  0  0  1  1  1  0  1  0  0  0  0
P11  0  0  0  0  1  0  1  0  1  1  0  0  0
P12  0  0  0  0  1  1  1  0  0  1  0  0  0
P13  0  0  0  0  1  1  1  1  1  0  0  0  0
P14  0  0  0  0  1  0  1  1  0  0  0  0  0
P15  0  0  0  0  1  1  0  1  0  1  0  0  0
P16  0  0  0  0  0  1  0  1  1  1  0  0  0
P17  0  0  0  0  0  0  0  0  0  0  1  1  1
P18  0  0  0  0  0  0  0  0  0  0  1  1  1
P19  0  0  0  0  0  0  0  0  0  0  0  1  1
P20  0  0  0  0  0  0  0  0  0  0  1  1  1


## DCA Algorithm Implementation

In [83]:
def dca_step1_initial_ordering(matrix):
    """
    Order rows and columns by sum of 1s
    """
    row_sums = matrix.sum(axis=1)
    col_sums = matrix.sum(axis=0)
    
    # Sort rows in DESCENDING order by sum
    # For ties, use descending order of original index
    row_order = row_sums.sort_values(ascending=False, kind='stable')
    sorted_rows = matrix.loc[row_order.index]
    
    col_order = col_sums.sort_values(ascending=True, kind='stable')
    sorted_matrix = sorted_rows[col_order.index]
    
    return sorted_matrix


def dca_step2_shift_columns_left(matrix):
    """
    Sort columns by shifting columns with 1s to the left
    - Starting with first row, shift all columns with 1 in that row to the left
    - Continue row by row until no further shifts possible
    """
    result = matrix.copy()
    cols = list(result.columns)
    
    shifted_cols = []
    remaining_cols = cols.copy()
    
    for row_idx in result.index:
        row = result.loc[row_idx]
        
        # Find columns with 1s in this row that haven't been shifted yet
        cols_with_ones = [col for col in remaining_cols if row[col] == 1]
        
        # Add these columns to the shifted list
        shifted_cols.extend(cols_with_ones)
        
        for col in cols_with_ones:
            remaining_cols.remove(col)
    
    # Add any remaining columns at the end
    shifted_cols.extend(remaining_cols)
    
    # Reorder the matrix with new column order
    result = result[shifted_cols]
    
    # print("\nStep 2: Column Left-Shifting")
    # print(f"New column order: {shifted_cols}")
    
    return result


def dca_step3_shift_rows_up(matrix):
    """
    Sort rows by shifting rows with 1s upward
    """
    result = matrix.copy()
    rows = list(result.index)
    
    shifted_rows = []
    remaining_rows = rows.copy()
    
    for col_name in result.columns:
        col = result[col_name]

        rows_with_ones = [row for row in remaining_rows if col[row] == 1]
        
        shifted_rows.extend(rows_with_ones)
        
        for row in rows_with_ones:
            remaining_rows.remove(row)
    
    shifted_rows.extend(remaining_rows)
    result = result.loc[shifted_rows]
    
    # print("\nStep 3: Row Upward-Shifting")
    # print(f"New row order: {shifted_rows}")
    
    return result


def dca_algorithm(matrix):

    print("="*80)
    print("DCA ALGORITHM - Direct Clustering Algorithm")
    print("="*80)
    
    matrix_step1 = dca_step1_initial_ordering(matrix)
    print("\nMatrix after Step 1:")
    print(matrix_step1)
    print("\n" + "="*80)
    
    matrix_step2 = dca_step2_shift_columns_left(matrix_step1)
    print("\nMatrix after Step 2:")
    print(matrix_step2)
    print("\n" + "="*80)
    
    matrix_step3 = dca_step3_shift_rows_up(matrix_step2)
    print("\nMatrix after Step 3 (FINAL):")
    print(matrix_step3)
    print("\n" + "="*80)
    
    return matrix_step3

In [84]:
# implement
dca_result = dca_algorithm(incidence_df)

DCA ALGORITHM - Direct Clustering Algorithm

Matrix after Step 1:
     K  A  L  M  B  D  C  F  G  H  E  I  J
P6   0  1  0  0  1  1  1  0  0  1  0  1  1
P1   0  1  0  0  1  1  1  0  0  0  0  1  1
P2   0  1  0  0  0  1  1  0  0  1  0  0  1
P3   0  0  0  0  1  1  1  0  0  0  0  1  1
P4   0  1  0  0  1  1  0  0  1  1  0  0  0
P7   0  0  0  0  0  0  1  1  0  0  1  1  1
P9   0  0  0  0  0  0  0  1  1  0  1  1  1
P13  0  0  0  0  0  0  0  1  1  1  1  1  0
P8   0  0  0  0  0  0  0  0  1  1  1  0  1
P10  0  0  0  0  0  0  0  1  1  0  1  1  0
P11  0  0  0  0  0  0  0  0  1  0  1  1  1
P12  0  0  0  0  0  0  0  1  1  0  1  0  1
P15  0  0  0  0  0  0  0  1  0  1  1  0  1
P16  0  0  0  0  0  0  0  1  0  1  0  1  1
P5   0  0  0  0  1  0  1  0  0  0  0  1  0
P14  0  0  0  0  0  0  0  0  1  1  1  0  0
P17  1  0  1  1  0  0  0  0  0  0  0  0  0
P18  1  0  1  1  0  0  0  0  0  0  0  0  0
P20  1  0  1  1  0  0  0  0  0  0  0  0  0
P19  0  0  1  1  0  0  0  0  0  0  0  0  0


Matrix after Step 2:
     A  

### Note: Use Steps rather than using Machines for the matrix

because:


To start with simplification, the __multi functional machines are not considered here__, which makes the capacity planning of them to be none. It is said to use steps as axis 1, which is rather use unifunctional machines as axis 1. This aligns with DCA purpooses

In [85]:
# Save the DCA reordered matrix
dca_result.to_csv('dca_reordered_matrix.csv')

in current repo, the raw output file is handled in excel for better visualization and other analysis and be saved as .xlsx file, feel free to download and check 